# Modelisation supervisee (classification + regression)

Objectif: entrainer des modeles pour predire `Categorie` (classification) et `Prix_Revente` (regression).

Donnees attendues: `data/processed/train_clean.csv`, `val_clean.csv`, `test_clean.csv`.


In [6]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor

try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    root_mean_squared_error = None

DATA_DIR = Path("..") / "data" / "processed"

train_df = pd.read_csv(DATA_DIR / "train_clean.csv")
val_df = pd.read_csv(DATA_DIR / "val_clean.csv")
test_df = pd.read_csv(DATA_DIR / "test_clean.csv")

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)


Train: (6449, 16)
Val: (1382, 16)
Test: (1382, 16)


## 1. Preparation des features et cibles

In [7]:
target_cls = "Categorie"
target_reg = "Prix_Revente"

text_cols = ["Rapport_Collecte"] if "Rapport_Collecte" in train_df.columns else []

# Retirer les colonnes non numeriques pour les modeles tabulaires classiques
feature_cols = [c for c in train_df.columns if c not in [target_cls, target_reg] + text_cols]

X_train = train_df[feature_cols]
y_train_cls = train_df[target_cls]
y_train_reg = train_df[target_reg]

X_val = val_df[feature_cols]
y_val_cls = val_df[target_cls]
y_val_reg = val_df[target_reg]

X_test = test_df[feature_cols]
y_test_cls = test_df[target_cls]
y_test_reg = test_df[target_reg]

print("Features:", len(feature_cols))

Features: 13


## 2. Classification (baseline + GridSearch)

On compare quelques modeles, puis on fait un tuning simple.

In [8]:
cls_models = {
    "logreg": LogisticRegression(max_iter=2000),
    "rf": RandomForestClassifier(random_state=42),
    "gb": GradientBoostingClassifier(random_state=42),
}

for name, model in cls_models.items():
    model.fit(X_train, y_train_cls)
    preds = model.predict(X_val)
    acc = accuracy_score(y_val_cls, preds)
    f1 = f1_score(y_val_cls, preds, average="weighted")
    print(name, "acc=", round(acc, 4), "f1=", round(f1, 4))

# GridSearch sur RandomForest
param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
}

rf = RandomForestClassifier(random_state=42)
search = GridSearchCV(rf, param_grid, cv=3, n_jobs=-1, scoring="f1_weighted")
search.fit(X_train, y_train_cls)

print("Best params:", search.best_params_)

best_cls = search.best_estimator_
val_preds = best_cls.predict(X_val)
print("Val report:\n", classification_report(y_val_cls, val_preds))


logreg acc= 0.9725 f1= 0.9725
rf acc= 0.9964 f1= 0.9964
gb acc= 0.9971 f1= 0.9971
Best params: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 400}
Val report:
               precision    recall  f1-score   support

       Métal       0.99      1.00      1.00       325
      Papier       1.00      0.99      0.99       316
   Plastique       0.99      1.00      1.00       384
       Verre       1.00      1.00      1.00       357

    accuracy                           1.00      1382
   macro avg       1.00      1.00      1.00      1382
weighted avg       1.00      1.00      1.00      1382



## 3. Regression (baseline + GridSearch)

On predit `Prix_Revente` et on compare les erreurs.

In [9]:
reg_models = {
    "linreg": LinearRegression(),
    "rf": RandomForestRegressor(random_state=42),
    "gb": GradientBoostingRegressor(random_state=42),
}

def _rmse(y_true, y_pred):
    if root_mean_squared_error is not None:
        return root_mean_squared_error(y_true, y_pred)
    return mean_squared_error(y_true, y_pred, squared=False)

for name, model in reg_models.items():
    model.fit(X_train, y_train_reg)
    preds = model.predict(X_val)
    mae = mean_absolute_error(y_val_reg, preds)
    rmse = _rmse(y_val_reg, preds)
    r2 = r2_score(y_val_reg, preds)
    print(name, "mae=", round(mae, 4), "rmse=", round(rmse, 4), "r2=", round(r2, 4))

param_grid_reg = {
    "n_estimators": [200, 400],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
}

rf_reg = RandomForestRegressor(random_state=42)
search_reg = GridSearchCV(rf_reg, param_grid_reg, cv=3, n_jobs=-1, scoring="neg_mean_absolute_error")
search_reg.fit(X_train, y_train_reg)

print("Best params (reg):", search_reg.best_params_)

best_reg = search_reg.best_estimator_
val_preds = best_reg.predict(X_val)
print("Val MAE:", mean_absolute_error(y_val_reg, val_preds))
print("Val RMSE:", _rmse(y_val_reg, val_preds))
print("Val R2:", r2_score(y_val_reg, val_preds))


linreg mae= 106.5396 rmse= 706.2501 r2= 0.0088
rf mae= 3.8258 rmse= 36.5059 r2= 0.9974
gb mae= 19.6276 rmse= 82.9354 r2= 0.9863
Best params (reg): {'max_depth': 20, 'min_samples_split': 5, 'n_estimators': 200}
Val MAE: 3.7740816075544337
Val RMSE: 37.06010355910547
Val R2: 0.9972707846974118


## 4. Evaluation finale sur le test

In [10]:
test_preds_cls = best_cls.predict(X_test)
print("Test ACC:", accuracy_score(y_test_cls, test_preds_cls))
print("Test F1:", f1_score(y_test_cls, test_preds_cls, average="weighted"))


test_preds_reg = best_reg.predict(X_test)
print("Test MAE:", mean_absolute_error(y_test_reg, test_preds_reg))
print("Test RMSE:", _rmse(y_test_reg, test_preds_reg))
print("Test R2:", r2_score(y_test_reg, test_preds_reg))


Test ACC: 0.9978292329956585
Test F1: 0.9978282521282784
Test MAE: 9.084523565666794
Test RMSE: 251.2743045188565
Test R2: 0.8246143269353529
